# ESM3 sequon scoring — the structure track, on and off

Runs the 2×2 that makes ESM3 worth having: **structure conditioning on/off**
crossed with **motif visible/hidden**, inside one model with one tokeniser and
one masking scheme.

Every other structure-versus-sequence comparison in this benchmark is *between*
models, so it confounds the question with architecture, training data and
tokenisation. This one does not.

| Variant | Structure track | Masking |
|---|---|---|
| `esm3_struct_single` | intact | scored position only |
| `esm3_struct_joint` | intact | all three sequon positions |
| `esm3_seq_single` | withheld | scored position only |
| `esm3_seq_joint` | withheld | all three sequon positions |

About seven hours on CPU; roughly twenty minutes on a GPU.

**The branch must be pushed first.** This clones from GitHub, so anything
uncommitted or unpushed locally will not be here.


## 1. GPU, dependencies, gated checkpoint


In [ ]:
!nvidia-smi -L


In [ ]:
# Verified on a clean Python 3.13 environment before being written here,
# because Colab runs 3.13 while ARC runs 3.12 and the two need different
# recipes. Three things this gets right that earlier versions did not:
#
#   esm==3.2.2 CANNOT install on Colab. It declares requires_python
#   >=3.12,<3.13, so the pin copied from the ARC setup is unsatisfiable here.
#   3.4.0 is the version that supports 3.13.
#
#   --no-deps on `esm` protects torch. esm declares torch<2.12,>=2.11, and
#   letting pip resolve that would replace Colab's CUDA-matched build --
#   worse than any missing package. Its OTHER dependencies then install
#   normally, because transformers validates its own deps at import and
#   --no-deps everywhere leaves it broken.
#
#   Not quiet, and verified by importing in a FRESH interpreter, so neither a
#   silent pip failure nor a stale import in this kernel can look like success.
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps',
                'esm==3.4.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'transformers>=4.57.6,<5.0.0', 'einops', 'biotite>=1.0.0',
                'msgpack-numpy', 'biopython', 'scikit-learn', 'brotli',
                'attrs', 'pandas', 'cloudpathlib', 'httpx', 'tenacity',
                'zstd', 'huggingface_hub', 'safetensors', 'pygtrie',
                'accelerate', 'ipython'], check=True)

# pip will warn that esm wants torch<2.12 and rdkit. Both are expected: torch
# is deliberately left alone, and rdkit is not on this code path.
probe = subprocess.run([sys.executable, '-c', '''
import torch, esm
from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein
from esm.utils.structure.protein_chain import ProteinChain
from esm.tokenization import EsmSequenceTokenizer
t = EsmSequenceTokenizer()
assert t.vocab_size == 33 and t.mask_token_id == 32
assert t.convert_tokens_to_ids("N") == 17
print(f"esm {esm.__version__} | torch {torch.__version__} | vocab OK")
'''], capture_output=True, text=True)
print(probe.stdout.strip() or probe.stderr.strip()[-2500:])
assert probe.returncode == 0, (
    'esm did not install usably. If the error names a package already imported\n'
    'by this kernel, use Runtime -> Restart session and run this cell FIRST.')


In [ ]:
# esm3-sm-open-v1 is gated: accept the licence at
# https://huggingface.co/EvolutionaryScale/esm3-sm-open-v1 first, then paste a token.
# Run this only after the one-time restart requested by the install cell.
# This import catches an in-memory mix of the old and newly installed Hub.
from huggingface_hub.utils import refresh_xet_connection_info
from huggingface_hub import login
login()


## 2. Repository and data

`results/` is gitignored, so the manifests are **not** in the repository. They
and the structures come from the same release bundle the ARC setup uses, which
keeps every environment reading identical inputs.


In [ ]:
import os, subprocess, sys
from pathlib import Path

BRANCH = 'fix/context-extractor-mapping'
REPO   = 'https://github.com/LBDillon/Glycan-occupancy-analysis.git'
BUNDLE = ('https://github.com/LBDillon/Glycan-occupancy-analysis/releases/'
          'download/bundle-2026-08-20/colab_bundle.tar')
MODULE = '/content/module'

# Fetch and reset rather than skipping when the directory exists. Restarting a
# Colab session restarts the kernel but keeps /content, so a checkout from an
# earlier run survives -- and a clone guarded only by directory existence then
# silently keeps stale code, which reads as 'no module named ...' for anything
# added since.
if os.path.exists(MODULE):
    subprocess.run(['git', '-C', MODULE, 'fetch', '--depth', '1', 'origin', BRANCH],
                   check=True)
    subprocess.run(['git', '-C', MODULE, 'reset', '--hard', f'origin/{BRANCH}'],
                   check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, MODULE],
                   check=True)

head = subprocess.run(['git', '-C', MODULE, 'log', '--oneline', '-1'],
                      capture_output=True, text=True).stdout.strip()
print('HEAD:', head)

# The whole point of this notebook is code that only exists on this branch, so
# check it arrived rather than discovering it three cells later.
required = ['src/experimental_glycosylation_sites/adapters/esm3.py',
            'src/experimental_glycosylation_sites/esm3_scoring.py']
missing = [f for f in required if not os.path.exists(f'{MODULE}/{f}')]
assert not missing, (f'{missing} absent from the checkout. The branch may not be '
                     'pushed, or this is a stale clone: rm -rf /content/module '
                     'and rerun.')
print('esm3 adapter present')
assert not head.startswith('aeff61d'), (
    'This is the stale pre-autocast checkout that failed on a T4. Rerun '
    'this cell after the branch has been pushed; do not patch the smoke '
    'model by hand because the full run starts a fresh process.')


In [ ]:
# Plain Python rather than shell magic: `find -exec` and IPython's brace
# substitution disagree about escaping, and a half-extracted bundle looks
# exactly like a complete one.
import gzip, shutil, tarfile, urllib.request

STRUCT = Path(MODULE) / 'data/cache/pdb'
STRUCT.mkdir(parents=True, exist_ok=True)

if not any(STRUCT.iterdir()):
    archive_path = Path('/content/colab_bundle.tar')
    if not archive_path.exists():
        print('downloading 594 MB...', flush=True)
        urllib.request.urlretrieve(BUNDLE, archive_path)
    with tarfile.open(archive_path) as archive:
        archive.extractall('/content/extracted')

    source = Path('/content/extracted')
    for path in (source / 'structures').rglob('*'):
        if path.suffix == '.gz':
            with gzip.open(path) as fh, open(STRUCT / path.stem, 'wb') as out:
                shutil.copyfileobj(fh, out)
        elif path.suffix in ('.pdb', '.cif'):
            shutil.copy(path, STRUCT / path.name)

    for kind in ('manifests', 'matching'):
        target = Path(MODULE) / 'results' / kind
        target.mkdir(parents=True, exist_ok=True)
        for path in (source / kind).glob('*.csv'):
            shutil.copy(path, target / path.name)

structures = list(STRUCT.iterdir())
manifests = list((Path(MODULE) / 'results/manifests').glob('*.csv'))
print('structures:', len(structures), '(expect 1824)')
print('manifests :', len(manifests))
assert structures and manifests, 'bundle did not extract; rerun this cell'


## 3. Verify before running anything long

Three chains whose sequon indices and triplets are established in
`docs/methods_sequon_indexing.md`. This checks the token offset, that ESM3's
parse agrees with the manifest's, and — the point of the model — that
withholding the structure track actually changes the answer.

Were `|struct − seq_only|` zero, the track would not really be off and every
number from the long run would be meaningless.


In [ ]:
import sys
sys.path.insert(0, f'{MODULE}/src')
os.chdir(MODULE)

from experimental_glycosylation_sites.adapters.esm3 import ESM3Adapter

base = ESM3Adapter(device='cuda')
model, tokenizer = base._load()
print('token offset verified; mask id', tokenizer.mask_token_id)

CASES = [('4EBY','A',27,'NSS'), ('5H5Y','A',226,'NRS'), ('9G3Q','A',181,'NES')]
for pdb, chain, n, triplet in CASES:
    path = STRUCT / f'{pdb}.pdb'
    if not path.exists():
        print(f'{pdb}: not in the bundle, skipped'); continue
    indices = (n, n + 1, n + 2)
    got = {}
    for structure_mode in ('struct_cond', 'seq_only'):
        adapter = ESM3Adapter(device='cuda', structure_mode=structure_mode,
                              mask_mode='single')
        adapter._model, adapter._tokenizer = model, tokenizer
        got[structure_mode] = adapter.score_site(str(path), chain, indices,
                                                 expected_triplet=triplet)
    a = got['struct_cond']['conditional_sequon_score']
    b = got['seq_only']['conditional_sequon_score']
    print(f'{pdb}:{chain}  struct {a:+.4f}  seq_only {b:+.4f}  |diff| {abs(a-b):.4f}')
    assert abs(a - b) > 1e-6, f'{pdb}: withholding structure changed nothing'
print('\nverified')


## 4. The 2×2

Both arms of each variant: `candidate_manifest_dataset` is the occupied set,
`manifest_matched_secretory` its matched partners. Stage 07 is resumable, so a
dropped runtime costs only the chain in flight — rerun this cell and it picks up.


In [ ]:
# Mounted BEFORE the long run, not after it. A Colab runtime can vanish at any
# point, and a save cell that only runs at the end saves nothing when that
# happens. Anything already in Drive is copied back first, so a fresh runtime
# resumes rather than restarting -- stage 07 skips sites already in its output.
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/glyco_occupancy/esm3')
OUT.mkdir(parents=True, exist_ok=True)

scores = Path(MODULE) / 'results/scores'
analysis = Path(MODULE) / 'results/analysis'
scores.mkdir(parents=True, exist_ok=True)
analysis.mkdir(parents=True, exist_ok=True)

restored = 0
for path in OUT.glob('*.csv'):
    target = analysis if path.name.startswith(('contrasts_', 'analysis_')) else scores
    if not (target / path.name).exists():
        shutil.copy(path, target / path.name)
        restored += 1
for path in OUT.glob('*.json'):
    if not (analysis / path.name).exists():
        shutil.copy(path, analysis / path.name)
        restored += 1
print(f'Drive: {len(list(OUT.iterdir()))} files, restored {restored} into the runtime')


def save_to_drive(note=''):
    """Copy every result out. Called after each arm, not just at the end."""
    copied = 0
    for pattern in ('results/scores/scores_*_esm3_*.csv',
                    'results/analysis/*esm3*.json',
                    'results/analysis/*esm3*.csv'):
        for path in Path(MODULE).glob(pattern):
            shutil.copy(path, OUT / path.name)
            copied += 1
    print(f'  saved {copied} files to Drive {note}'.rstrip())


In [ ]:
VARIANTS = [
    ('esm3_struct_single', 'struct_cond', 'single'),
    ('esm3_struct_joint',  'struct_cond', 'joint'),
    ('esm3_seq_single',    'seq_only',    'single'),
    ('esm3_seq_joint',     'seq_only',    'joint'),
]
ARMS = [('dataset', 'candidate_manifest_dataset'),
        ('secretory', 'manifest_matched_secretory')]


def score(variant, structure_mode, mask_mode, tag, manifest):
    '''Run one arm, and on failure show what the child actually said.

    A raised CalledProcessError names the command and nothing else, which is how
    three earlier failures here arrived: the cause sat in the subprocess output
    and never reached the reader.
    '''
    done = subprocess.run(
        [sys.executable, 'pipeline/07_score.py',
         f'results/manifests/{manifest}.csv',
         f'results/scores/scores_{tag}_{variant}.csv',
         '--model', 'esm3', '--structure-mode', structure_mode,
         '--mask-mode', mask_mode, '--device', 'auto'],
        capture_output=True, text=True)
    print(done.stdout[-3000:] or '(no stdout)')
    if done.returncode:
        print('--- STDERR ---')
        print(done.stderr[-4000:] or '(no stderr)')
        raise SystemExit(f'{variant}/{tag} exited {done.returncode}; the cause is above')


# Saved after EVERY arm, so a runtime that dies mid-sweep costs the arm in
# flight rather than the sweep. Rerunning resumes: stage 07 skips sites already
# present in its output.
for variant, structure_mode, mask_mode in VARIANTS:
    for tag, manifest in ARMS:
        print(f'\n=== {variant} / {tag} ===', flush=True)
        score(variant, structure_mode, mask_mode, tag, manifest)
        save_to_drive(f'after {variant}/{tag}')


## 5. Contrasts

The four conditioning strings are distinct, so no two arms can be pooled by
accident. `struct` minus `seq` is the within-model structure contribution.


In [ ]:
for variant, _, _ in VARIANTS:
    print(f'\n===== {variant} =====', flush=True)
    subprocess.run([sys.executable, '-u', 'pipeline/09_analyse_scores.py',
                    'secretory', '--variant', variant], check=True,
                   env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    expected = Path(f'results/analysis/analysis_secretory_{variant}.json')
    assert expected.exists(), f'analysis did not write {expected}'


In [ ]:
import json
import pandas as pd

rows = []
for variant, structure_mode, mask_mode in VARIANTS:
    path = Path(f'results/analysis/analysis_secretory_{variant}.json')
    if not path.exists():
        continue
    d = json.loads(path.read_text())
    rows.append({'variant': variant, 'structure': structure_mode,
                 'masking': mask_mode,
                 'effect_SD': round(d['mean_difference_sd'], 3),
                 'ci_low': round(d['ci95_sd'][0], 3),
                 'ci_high': round(d['ci95_sd'][1], 3),
                 'n': d['n_contrasts'], 'verdict': d['verdict']})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

assert len(summary) == 4, f'expected four completed variants, found {len(summary)}'
effect = summary.set_index('variant').effect_SD
visible = effect['esm3_struct_single'] - effect['esm3_seq_single']
hidden = effect['esm3_struct_joint'] - effect['esm3_seq_joint']
print(f'\nstructure contribution, motif visible: {visible:+.3f} SD')
print(f'structure contribution, motif hidden : {hidden:+.3f} SD')
print('\nPoint estimates of a difference of differences. Their interval is',
      'NOT the difference of the two intervals above, and computing it',
      'properly needs the paired contrasts, not these summaries.')


## 6. Save

Colab runtimes are ephemeral. Copy the scores and analyses out before the
session ends, then bring them down and rerun the figures locally.


In [ ]:
save_to_drive('final')
print(sorted(p.name for p in OUT.iterdir()))
